In [ ]:
from pathlib import Path

import numpy as np
import pyvista as pv

from ulac.common import mesh_utils, visualization

In [ ]:
patient_id = "02"
input_mesh_file = Path(f"../data/patient_{patient_id}/mesh_endo_raw.vtu")
output_mesh_file = Path("test_mesh.vtu")
decimation_factor = 1
num_smoothing_iterations = 0
extraction_threshold = 0.75
dimension_scaling_factor = 1e-3
legacy_tags = {"MV": 11, "LAA": 13, "LIPV": 21, "LSPV": 23, "RIPV": 25, "RSPV": 27}
new_tags = {"MV": 0, "LAA": 1, "LIPV": 2, "LSPV": 3, "RIPV": 4, "RSPV": 5}

In [ ]:
input_mesh = pv.read(input_mesh_file)
triangular_mesh = mesh_utils.convert_unstructured_to_polydata_mesh(input_mesh)
del triangular_mesh.point_data["alpha"]
del triangular_mesh.point_data["beta"]
for key, legacy_value in legacy_tags.items():
    anatomical_region = np.where(triangular_mesh.cell_data["anatomical_tags"] == legacy_value)[0]
    triangular_mesh.cell_data["anatomical_tags"][anatomical_region] = new_tags[key]

triangular_mesh = mesh_utils.preprocess_mesh(
    triangular_mesh,
    tag_values=new_tags.values(),
    decimation_factor=decimation_factor,
    num_smoothing_iters=num_smoothing_iterations,
    extraction_threshold=extraction_threshold,
    dimension_scaling_factor=dimension_scaling_factor,
)
output_mesh_file.parent.mkdir(parents=True, exist_ok=True)
pv.save_meshio(output_mesh_file, triangular_mesh)

In [ ]:
visualization.plot_mesh_with_vector_field(
    input_mesh,
    vector_field=input_mesh.cell_data["fibers"],
    scaling_factor=500,
)

In [ ]:
plotter = pv.Plotter(window_size=[900, 900])
plotter.add_mesh(
    input_mesh,
    scalars="fibers",
    cmap="tab20",
    show_edges=True,
    edge_color="lightgray",
    edge_opacity=0.3,
    show_scalar_bar=False,
)
plotter.show()